In [ ]:
# === PAPERMILL PARAMETERS ===
# Tag this cell with: parameters

# File paths
BASE_FOLDER = "data"
MASTER_CSV = "data/Betfair_Master.csv"
GOOGLE_SERVICE_ACCOUNT_JSON = ".secrets/google_service_account.json"

# Google Sheets
GOOGLE_SHEET_NAME = "Betfair Dashboard"

# Analysis settings
ROLLING_START_DATE = "2025-03-01"
WEEK_START_DAY = "W-SUN"
TOP_N_TRACKS = 15
TOP_N_STRIKE_RATES = 10

<a href="https://colab.research.google.com/github/gazuty/betfair-dashboard/blob/colab-stable-2025-08-10/betfair_dashboard_STABLE_2025_08_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === STEP 0: Configuration & Utilities ===

# --- Standard library imports ---
import glob
import os
import shutil
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

# --- Third-party imports ---
import pandas as pd
import pytz

# --- Colab detection and Drive mounting ---
try:
    IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")
    print("✅ Running in Google Colab with Drive mounted")
else:
    print("✅ Running locally")

# --- Resolve paths from parameters ---
BASE_FOLDER = os.getenv("BASE_FOLDER", BASE_FOLDER)
MASTER_CSV = os.getenv("MASTER_CSV", MASTER_CSV)
GOOGLE_SHEET_NAME = os.getenv("GOOGLE_SHEET_NAME", GOOGLE_SHEET_NAME)
GOOGLE_SERVICE_ACCOUNT_JSON = os.getenv(
    "GOOGLE_SERVICE_ACCOUNT_JSON", GOOGLE_SERVICE_ACCOUNT_JSON
)

ARCHIVE_FOLDER = str(Path(BASE_FOLDER) / "Archive")
BETTING_PATTERN = str(Path(BASE_FOLDER) / "BettingPandL*.csv")

# --- Business rules ---
VALID_SPORTS = ["Horse Racing", "Greyhound Racing"]
MIN_STRIKE_BETS = 50

# --- Ensure directories exist ---
os.makedirs(ARCHIVE_FOLDER, exist_ok=True)
if MASTER_CSV:
    master_dir = os.path.dirname(MASTER_CSV)
    if master_dir:
        os.makedirs(master_dir, exist_ok=True)


# === UTILITY FUNCTIONS ===

def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize date and profit columns for all Betfair data.

    Args:
        df: Raw DataFrame with 'Settled date' and 'Profit_Loss' columns

    Returns:
        DataFrame with parsed dates and numeric profit values
    """
    df = df.copy()
    df["Settled date"] = pd.to_datetime(df["Settled date"], errors="coerce")
    df["Profit_Loss"] = pd.to_numeric(df["Profit_Loss"], errors="coerce")
    return df.dropna(subset=["Settled date"]).reset_index(drop=True)


def load_master(path: str) -> pd.DataFrame:
    """
    Load existing master CSV or return empty DataFrame.

    Args:
        path: Path to the master CSV file

    Returns:
        DataFrame with standardized columns
    """
    if not os.path.exists(path):
        print("ℹ️ No existing master found — starting fresh")
        return pd.DataFrame(columns=["Market", "Settled date", "Profit_Loss"])
    df = pd.read_csv(path)
    df = prepare_dataframe(df)
    print(f"✅ Loaded master ({len(df)} rows)")
    return df


def parse_raw_file(filepath: str) -> Optional[pd.DataFrame]:
    """
    Parse a single Betfair P&L export file.

    Args:
        filepath: Path to the raw CSV file

    Returns:
        DataFrame with standardized columns, or None if file is invalid
    """
    fname = os.path.basename(filepath)
    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        print(f"⚠️ Could not read {fname}: {e}")
        return None

    required = {"Market", "Settled date"}
    if not required.issubset(df.columns):
        missing = required - set(df.columns)
        print(f"⚠️ {fname} missing columns: {missing}")
        return None

    profit_cols = [c for c in df.columns if "profit" in c.lower()]
    if not profit_cols:
        print(f"⚠️ {fname} has no profit column")
        return None

    # Prefer AUD column if available
    pick = next((c for c in profit_cols if "aud" in c.lower()), profit_cols[0])

    result = prepare_dataframe(
        pd.DataFrame({
            "Market": df["Market"],
            "Settled date": df["Settled date"],
            "Profit_Loss": df[pick],
        })
    )
    print(f"📄 {fname} → {len(result)} rows from '{pick}'")
    return result


def deduplicate_bets(master: pd.DataFrame, new: pd.DataFrame) -> pd.DataFrame:
    """
    Return rows from new that don't already exist in master.

    Args:
        master: Existing master DataFrame
        new: New data to deduplicate

    Returns:
        DataFrame containing only unique new rows
    """
    if master.empty:
        return new
    if new.empty:
        return new

    master_hash = pd.util.hash_pandas_object(
        master[["Market", "Settled date", "Profit_Loss"]], index=False
    )
    new_hash = pd.util.hash_pandas_object(
        new[["Market", "Settled date", "Profit_Loss"]], index=False
    )
    return new[~new_hash.isin(master_hash)]


def archive_file(filepath: str, archive_folder: str) -> None:
    """
    Move a processed file to the archive folder.

    Args:
        filepath: Path to file to archive
        archive_folder: Destination folder
    """
    os.makedirs(archive_folder, exist_ok=True)
    dest = os.path.join(archive_folder, os.path.basename(filepath))
    shutil.move(filepath, dest)
    print(f"📦 Archived {os.path.basename(filepath)}")


# --- Print configuration summary ---
print("\n✅ Configuration loaded:")
print(f"  BASE_FOLDER               = {BASE_FOLDER}")
print(f"  MASTER_CSV                = {MASTER_CSV}")
print(f"  ARCHIVE_FOLDER            = {ARCHIVE_FOLDER}")
print(f"  BETTING_PATTERN           = {BETTING_PATTERN}")
print(f"  GOOGLE_SHEET_NAME         = {GOOGLE_SHEET_NAME}")
print(f"  GOOGLE_SERVICE_ACCOUNT_JSON = {GOOGLE_SERVICE_ACCOUNT_JSON}")
print(f"  VALID_SPORTS              = {VALID_SPORTS}")
print(f"  MIN_STRIKE_BETS           = {MIN_STRIKE_BETS}")
print(f"  ROLLING_START_DATE        = {ROLLING_START_DATE}")
print(f"  TOP_N_TRACKS              = {TOP_N_TRACKS}")
print(f"  TOP_N_STRIKE_RATES        = {TOP_N_STRIKE_RATES}")

In [ ]:
# === STEP 1: Update Master Data ===

# Cell execution guard
if "prepare_dataframe" not in dir():
    raise RuntimeError("⚠️ Run STEP 0 (Configuration & Utilities) first")


def update_betfair_master() -> pd.DataFrame:
    """
    Merge new Betfair P&L exports into the master CSV.

    Reads all files matching BETTING_PATTERN, deduplicates against
    the existing master, appends new records, and archives processed files.

    Returns:
        Updated master DataFrame
    """
    print("🔄 Starting master update...")

    # Load existing master
    master = load_master(MASTER_CSV)

    # Find raw files
    raw_files = glob.glob(BETTING_PATTERN)
    print(f"📂 Found {len(raw_files)} raw file(s)")

    if not raw_files:
        print("ℹ️ No new files to process")
        return master

    # Parse all raw files
    all_new: List[pd.DataFrame] = []
    for filepath in raw_files:
        parsed = parse_raw_file(filepath)
        if parsed is not None and not parsed.empty:
            all_new.append(parsed)

    if not all_new:
        print("ℹ️ No valid data found in new files")
        return master

    # Combine and deduplicate
    combined_new = pd.concat(all_new, ignore_index=True)
    unique_new = deduplicate_bets(master, combined_new)
    print(f"✅ {len(unique_new)} unique new row(s) identified")

    if unique_new.empty:
        print("ℹ️ No new unique bets found — master unchanged")
        # Still archive the files since they were processed
        for filepath in raw_files:
            archive_file(filepath, ARCHIVE_FOLDER)
        return master

    # Merge and save
    master = pd.concat([master, unique_new], ignore_index=True)
    master = master.sort_values("Settled date").reset_index(drop=True)
    master.to_csv(MASTER_CSV, index=False)
    print(f"✅ Master updated ({len(master)} rows) → {MASTER_CSV}")

    # Archive processed files
    for filepath in raw_files:
        archive_file(filepath, ARCHIVE_FOLDER)

    return master


# Run the update
df_master = update_betfair_master()

In [ ]:
# === STEP 2: Load Master Data ===

# Cell execution guard
if "MASTER_CSV" not in dir():
    raise RuntimeError("⚠️ Run STEP 0 (Configuration & Utilities) first")

print(f"📂 Loading master from: {MASTER_CSV}")

# Load and prepare data using unified function
df = prepare_dataframe(pd.read_csv(MASTER_CSV))

print(f"✅ {len(df)} rows loaded")
print(f"   Date range: {df['Settled date'].min().date()} to {df['Settled date'].max().date()}")
print(f"   Profit_Loss dtype: {df['Profit_Loss'].dtype}")

In [ ]:
# === STEP 3: Feature Extraction ===

# Cell execution guard
if "df" not in dir():
    raise RuntimeError("⚠️ Run STEP 2 (Load Master Data) first")

# Extract Sport from Market (first token before slash)
df["Sport"] = df["Market"].str.extract(r"^([^/]+)/")[0].str.strip()

# Extract Track_Info and Event_Description for racing sports
racing_mask = df["Sport"].isin(VALID_SPORTS)
track_event = df.loc[racing_mask, "Market"].str.extract(r"/\s*(.*?)\s*:\s*(.*)")
track_event.columns = ["Track_Info", "Event_Description"]
df.loc[racing_mask, ["Track_Info", "Event_Description"]] = track_event

# Extract Country from parentheses in Track_Info
df["Country"] = df["Track_Info"].str.extract(r"\(([^)]+)\)")[0]

# Fill missing country values
df["Country"] = df["Country"].fillna("UK")
df.loc[~df["Sport"].isin(VALID_SPORTS), "Country"] = "Unknown"

# Clean up Track_Info to produce Track_Name (remove dates and country)
df["Track_Name"] = (
    df["Track_Info"]
    .str.replace(r"\([^)]*\)", "", regex=True)
    .str.replace(r"\b\d{1,2}(?:st|nd|rd|th)?\s+\w+\b", "", regex=True)
    .str.strip()
)

# Preview output
preview = (
    df.loc[df["Track_Name"].notna(), ["Sport", "Track_Name", "Country"]]
    .drop_duplicates()
    .head(10)
)
print("✅ Feature extraction complete — sample tracks:")
print(preview)

In [ ]:
# === STEP 4: Build Summary Tables ===

# Cell execution guard
if "df" not in dir() or "Sport" not in df.columns:
    raise RuntimeError("⚠️ Run STEP 3 (Feature Extraction) first")

print("🔧 Building summary tables...")

# --- Helper function for rolling calculations ---
def compute_rolling_returns(
    df_input: pd.DataFrame,
    start_date: str,
    date_col: str = "Day",
    value_col: str = "Profit_Loss",
) -> pd.DataFrame:
    """
    Compute rolling 2w, 4w, and 8w returns from daily profit data.

    Args:
        df_input: DataFrame with daily profit data
        start_date: Start date for rolling window output
        date_col: Name of date column
        value_col: Name of value column

    Returns:
        DataFrame with rolling return columns
    """
    df_work = df_input.copy()
    df_work[date_col] = pd.to_datetime(df_work[date_col])
    df_work = df_work.set_index(date_col)

    result = pd.DataFrame(index=df_work.index)
    result["Rolling 2w"] = df_work[value_col].rolling(window="14D").sum()
    result["Rolling 4w"] = df_work[value_col].rolling(window="28D").sum()
    result["Rolling 8w"] = df_work[value_col].rolling(window="56D").sum()

    result = result.reset_index()
    result = result[result[date_col] >= pd.to_datetime(start_date)].round(2)
    return result


# Daily Summary (chronological)
by_day = (
    df.groupby(df["Settled date"].dt.date)["Profit_Loss"]
    .sum()
    .reset_index(name="Profit_Loss")
    .rename(columns={"Settled date": "Day"})
    .sort_values("Day")
    .reset_index(drop=True)
)
by_day["Cumulative_Profit_Loss"] = by_day["Profit_Loss"].cumsum()
by_day[["Profit_Loss", "Cumulative_Profit_Loss"]] = by_day[
    ["Profit_Loss", "Cumulative_Profit_Loss"]
].round(2)
by_day["Day"] = pd.to_datetime(by_day["Day"])

# Rolling Returns (overall)
rolling_df = compute_rolling_returns(by_day, ROLLING_START_DATE)
rolling_df.columns = ["Day", "Rolling 2w", "Rolling 4w", "Rolling 8w"]

# Weekly Summary
by_week = (
    df.set_index("Settled date")
    .resample(WEEK_START_DAY)["Profit_Loss"]
    .sum()
    .reset_index()
    .rename(columns={"Settled date": "Week Starting"})
)
by_week["Profit_Loss"] = by_week["Profit_Loss"].round(2)

# Rolling by Sport
rolling_by_sport: Dict[str, pd.DataFrame] = {}
for sport in VALID_SPORTS:
    sport_df = df[df["Sport"] == sport].copy()
    sport_by_day = (
        sport_df.groupby(sport_df["Settled date"].dt.date)["Profit_Loss"]
        .sum()
        .reset_index(name="Profit_Loss")
        .rename(columns={"Settled date": "Day"})
        .sort_values("Day")
        .reset_index(drop=True)
    )
    rolling_by_sport[sport] = compute_rolling_returns(sport_by_day, ROLLING_START_DATE)
    rolling_by_sport[sport].columns = ["Day", "Rolling 2w", "Rolling 4w", "Rolling 8w"]

# Monthly Summary (using 'ME' to avoid deprecation warning)
by_month = (
    df.set_index("Settled date").resample("ME")["Profit_Loss"].sum().reset_index()
)
by_month["Month"] = by_month["Settled date"].dt.to_period("M").astype(str)
by_month = by_month[["Month", "Profit_Loss"]]
by_month["Profit_Loss"] = by_month["Profit_Loss"].round(2)

# Sport Summary
by_sport = (
    df.groupby("Sport")["Profit_Loss"].sum().reset_index().round({"Profit_Loss": 2})
)

# Country Summary
by_country = (
    df.groupby("Country")["Profit_Loss"].sum().reset_index().round({"Profit_Loss": 2})
)

# Daily Summaries per Sport (with cumulative P/L)
sport_daily: Dict[str, pd.DataFrame] = {}
for sport in df["Sport"].dropna().unique():
    temp = (
        df[df["Sport"] == sport]
        .groupby(df["Settled date"].dt.date)["Profit_Loss"]
        .sum()
        .reset_index(name="Profit_Loss")
        .rename(columns={"Settled date": "Day"})
        .sort_values("Day")
        .reset_index(drop=True)
    )
    temp["Cumulative_Profit_Loss"] = temp["Profit_Loss"].cumsum().round(2)
    temp["Profit_Loss"] = temp["Profit_Loss"].round(2)
    sport_daily[f"{sport} Daily"] = temp

# Summary checks
print(f"✅ By Day: {len(by_day)} rows (last: {by_day['Day'].max().date()})")
print(f"✅ Rolling Returns: {len(rolling_df)} rows")
for k, v in rolling_by_sport.items():
    print(f"✅ Rolling {k}: {len(v)} rows")
print(f"✅ By Week: {len(by_week)} rows")
print(f"✅ By Month: {len(by_month)} rows")
print(f"✅ By Sport: {len(by_sport)} sports")
print(f"✅ By Country: {len(by_country)} countries")

In [ ]:
# === STEP 5: Track Summaries ===

# Cell execution guard
if "df" not in dir() or "Track_Name" not in df.columns:
    raise RuntimeError("⚠️ Run STEP 3 (Feature Extraction) first")

# Aggregate P/L per track for racing sports
track_df = (
    df[df["Sport"].isin(VALID_SPORTS)]
    .groupby(["Sport", "Track_Name"], as_index=False)["Profit_Loss"]
    .sum()
)
track_df["Profit_Loss"] = track_df["Profit_Loss"].round(2)

# Create summary groups using TOP_N_TRACKS constant
tracks: Dict[str, pd.DataFrame] = {
    "Track Stats": track_df,
    "Top Horse Tracks": track_df.query("Sport == 'Horse Racing'").nlargest(
        TOP_N_TRACKS, "Profit_Loss"
    ),
    "Bottom Horse Tracks": track_df.query("Sport == 'Horse Racing'").nsmallest(
        TOP_N_TRACKS, "Profit_Loss"
    ),
    "Top Greyhound Tracks": track_df.query("Sport == 'Greyhound Racing'").nlargest(
        TOP_N_TRACKS, "Profit_Loss"
    ),
    "Bottom Greyhound Tracks": track_df.query("Sport == 'Greyhound Racing'").nsmallest(
        TOP_N_TRACKS, "Profit_Loss"
    ),
}

# Preview
print(f"✅ Track summaries built (top/bottom {TOP_N_TRACKS} each)")
print("Sample Top Horse Tracks:")
print(tracks["Top Horse Tracks"][["Track_Name", "Profit_Loss"]].head())

In [ ]:
# === STEP 6: Strike Rates ===

# Cell execution guard
if "df" not in dir() or "Track_Name" not in df.columns:
    raise RuntimeError("⚠️ Run STEP 3 (Feature Extraction) first")

# Filter to racing sports
df_racing = df[df["Sport"].isin(VALID_SPORTS)].copy()

# Compute total bets and wins per track
strike_df = (
    df_racing.groupby(["Sport", "Track_Name"])["Profit_Loss"]
    .agg(total_bets="count", wins=lambda x: (x > 0).sum())
    .reset_index()
)

# Calculate strike rate
strike_df["Strike_Rate"] = (strike_df["wins"] / strike_df["total_bets"]).round(4)

# Filter by minimum bets threshold
strike_df_filtered = strike_df[
    strike_df["total_bets"] >= MIN_STRIKE_BETS
].reset_index(drop=True)

# Extract Top & Bottom Strike Rate Tracks using constant
top_strike = strike_df_filtered.nlargest(TOP_N_STRIKE_RATES, "Strike_Rate").reset_index(
    drop=True
)
bottom_strike = strike_df_filtered.nsmallest(
    TOP_N_STRIKE_RATES, "Strike_Rate"
).reset_index(drop=True)

# Preview
print(f"✅ Strike rates computed (min {MIN_STRIKE_BETS} bets, top/bottom {TOP_N_STRIKE_RATES})")
print(f"\nTop {TOP_N_STRIKE_RATES} Strike Rates:")
print(top_strike[["Sport", "Track_Name", "total_bets", "wins", "Strike_Rate"]])
print(f"\nBottom {TOP_N_STRIKE_RATES} Strike Rates:")
print(bottom_strike[["Sport", "Track_Name", "total_bets", "wins", "Strike_Rate"]])

In [ ]:
# === STEP 7: Prepare Export Data ===

# Cell execution guard
required_vars = ["by_day", "by_week", "by_month", "by_sport", "by_country", "tracks"]
missing = [v for v in required_vars if v not in dir()]
if missing:
    raise RuntimeError(f"⚠️ Missing variables: {missing}. Run previous steps first.")

# Initialize export dictionary
all_sheets: Dict[str, pd.DataFrame] = {}

# Core summaries
all_sheets.update({
    "By Day": by_day,
    "By Day Sorted": by_day.sort_values("Profit_Loss", ascending=False).reset_index(
        drop=True
    ),
    "By Week": by_week,
    "Cumulative": by_day[["Day", "Cumulative_Profit_Loss"]].rename(
        columns={"Cumulative_Profit_Loss": "Cumulative"}
    ),
    "By Month": by_month,
    "By Sport": by_sport,
    "By Country": by_country,
    "Rolling Returns": rolling_df,
})

# Track-level summaries
all_sheets.update({
    "Track Stats": tracks["Track Stats"],
    "Top Horse Tracks": tracks["Top Horse Tracks"],
    "Bottom Horse Tracks": tracks["Bottom Horse Tracks"],
    "Top Greyhound Tracks": tracks["Top Greyhound Tracks"],
    "Bottom Greyhound Tracks": tracks["Bottom Greyhound Tracks"],
})

# Strike rate summaries
all_sheets.update({
    "Top Strike Rates": top_strike,
    "Bottom Strike Rates": bottom_strike,
})

# Daily summaries for each sport
all_sheets.update(sport_daily)

# Rolling returns by sport
for sport, df_rolling in rolling_by_sport.items():
    all_sheets[f"Rolling {sport}"] = df_rolling

# Summary
print(f"✅ Prepared {len(all_sheets)} tables for export:")
for name in all_sheets:
    print(f"  • {name}")

In [ ]:
# === STEP 7b: Preview Track Data ===

print("📊 Top Horse Tracks preview:")
print(tracks["Top Horse Tracks"].head())
print("\n📊 Bottom Horse Tracks preview:")
print(tracks["Bottom Horse Tracks"].head())

In [ ]:
# === STEP 7c: Validation Check ===

# Cell execution guard
if "df" not in dir():
    raise RuntimeError("⚠️ Run STEP 2 (Load Master Data) first")

# Compare master totals with any raw files present
au = pytz.timezone("Australia/Sydney")
yesterday = (pd.Timestamp.now(au) - timedelta(days=1)).date()
today = pd.Timestamp.now(au).date()

# Master totals (df is already prepared, no need to re-parse)
mask_yesterday = df["Settled date"].dt.date == yesterday
mask_today = df["Settled date"].dt.date == today

master_yesterday = float(df.loc[mask_yesterday, "Profit_Loss"].sum())
master_today = float(df.loc[mask_today, "Profit_Loss"].sum())

# Raw totals (if any raws exist)
raw_yesterday, raw_today = 0.0, 0.0
raw_files = glob.glob(BETTING_PATTERN)

if raw_files:
    raws: List[pd.DataFrame] = []
    for f in raw_files:
        try:
            r = pd.read_csv(f)
            if {"Settled date", "Profit_Loss"}.issubset(r.columns):
                r = prepare_dataframe(r)
                raws.append(r[["Settled date", "Profit_Loss"]])
        except Exception:
            pass

    if raws:
        R = pd.concat(raws, ignore_index=True)
        raw_yesterday = float(R.loc[R["Settled date"].dt.date == yesterday, "Profit_Loss"].sum())
        raw_today = float(R.loc[R["Settled date"].dt.date == today, "Profit_Loss"].sum())

print(f"MASTER → {yesterday}: {master_yesterday:.2f} | {today}: {master_today:.2f}")
print(f" RAWS  → {yesterday}: {raw_yesterday:.2f} | {today}: {raw_today:.2f}  (0.00 if no raw files)")

# Validation pass/fail
ok = (
    (abs(master_yesterday - raw_yesterday) < 0.01 or raw_yesterday == 0.0)
    and (abs(master_today - raw_today) < 0.01 or raw_today == 0.0)
)
print("PASS ✅ — proceed to Step 8" if ok else "STOP ❌ — mismatch detected")

In [ ]:
# === STEP 8: Export to Google Sheets ===

# Cell execution guard
if "all_sheets" not in dir():
    raise RuntimeError("⚠️ Run STEP 7 (Prepare Export Data) first")

import gspread
from gspread.exceptions import APIError, WorksheetNotFound


def values_from_df(df: pd.DataFrame) -> List[List[str]]:
    """
    Convert DataFrame to list of lists for Google Sheets upload.

    Args:
        df: DataFrame to convert

    Returns:
        List of lists with header row and data rows as strings
    """
    out = df.copy()
    for c in out.select_dtypes(include=["float", "int"]).columns:
        out[c] = pd.to_numeric(out[c], errors="coerce").round(2)
    return [out.columns.tolist()] + out.fillna("").astype(str).values.tolist()


def retry_gs(fn: Callable, *args: Any, **kwargs: Any) -> Any:
    """
    Retry a Google Sheets API call with exponential backoff.

    Args:
        fn: Function to call
        *args: Positional arguments for fn
        **kwargs: Keyword arguments for fn

    Returns:
        Result of fn call
    """
    delay = 3
    for attempt in range(6):
        try:
            return fn(*args, **kwargs)
        except APIError as e:
            if "429" in str(e):
                print(f"⏳ Rate limited; retrying in {delay}s...")
                time.sleep(delay)
                delay = min(delay * 2, 30)
            else:
                raise
    raise RuntimeError("❌ Max retries exceeded for Google Sheets API")


def safe_gspread_connect(service_json: str) -> gspread.Client:
    """
    Connect to Google Sheets with helpful error messages.

    Args:
        service_json: Path to service account JSON file

    Returns:
        Authenticated gspread client
    """
    if not os.path.exists(service_json):
        raise FileNotFoundError(
            f"❌ Service account JSON not found at: {service_json}\n"
            "Download from: Google Cloud Console → IAM → Service Accounts"
        )
    try:
        return gspread.service_account(filename=service_json)
    except Exception as e:
        raise ConnectionError(f"❌ Failed to authenticate with Google: {e}")


# Connect to Google Sheets
gc = safe_gspread_connect(GOOGLE_SERVICE_ACCOUNT_JSON)
sh = retry_gs(gc.open, GOOGLE_SHEET_NAME)
print(f"✅ Connected to Google Sheet: {GOOGLE_SHEET_NAME}")

# Write all tables
for sheet_name, df_out in all_sheets.items():
    try:
        ws = sh.worksheet(sheet_name)
    except WorksheetNotFound:
        ws = retry_gs(sh.add_worksheet, title=sheet_name, rows=200, cols=26)

    vals = values_from_df(df_out)
    retry_gs(ws.clear)
    retry_gs(ws.update, vals, range_name="A1", value_input_option="RAW")
    print(f"✅ Updated: {sheet_name} ({len(df_out)} rows)")

# Dashboard KPIs
total_profit = float(df["Profit_Loss"].sum())
total_bets = len(df)

best_day_row = by_day.loc[by_day["Profit_Loss"].idxmax()]
worst_day_row = by_day.loc[by_day["Profit_Loss"].idxmin()]
best_day = f'{best_day_row["Day"].date()} ({best_day_row["Profit_Loss"]:.2f})'
worst_day = f'{worst_day_row["Day"].date()} ({worst_day_row["Profit_Loss"]:.2f})'

kpis = [
    ["Metric", "Value"],
    ["Total Profit/Loss", round(total_profit, 2)],
    ["Number of Bets", total_bets],
    ["Best Day", best_day],
    ["Worst Day", worst_day],
    ["Generated on", str(date.today())],
]

try:
    dash = sh.worksheet("Dashboard")
except WorksheetNotFound:
    dash = retry_gs(sh.add_worksheet, title="Dashboard", rows=50, cols=6)

retry_gs(dash.update, kpis, range_name="A1", value_input_option="RAW")
print("✅ Dashboard KPIs updated")
print(f"\n🎉 Export complete! {len(all_sheets)} sheets updated.")